<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>AI SYSTEMS ENGINEERING 2</b>

## Unit 2: Agents, Tools, and Handoffs

**CV Raman Global University, Bhubaneswar**
*AI Center of Excellence*

</div>

---

### What You'll Learn

In this notebook, you will:

1. **Build your first agent** — create an agent, run it, and trace its execution
2. **Give agents tools** — use @function_tool to let agents call Python functions
3. **Compose agents** — use agent.as_tool() and asyncio.gather() for parallel work
4. **Hand off between agents** — understand when to use tools vs handoffs
5. **Orchestrate a multi-step workflow** — combine everything into a working pipeline

## 1. Environment Setup

In [ ]:
!pip install -q openai-agents

import os
import asyncio
from getpass import getpass
from agents import Agent, Runner, trace, function_tool

In [ ]:
api_key = getpass("Enter your OpenAI API Key: ")
os.environ['OPENAI_API_KEY'] = api_key
MODEL = "gpt-4o-mini"

---

## 2. The Problem — Why Agent Frameworks?

Building an LLM-powered agent from scratch means writing a lot of boilerplate: parsing tool calls, managing conversation history, handling retries, routing between models. The OpenAI Agents SDK handles all of this in a lightweight package.

| DIY Agent Loop | Agents SDK |
|---|---|
| Parse tool call JSON manually | `@function_tool` auto-generates schema |
| Track conversation state yourself | `Runner` manages the loop |
| Build your own routing logic | `handoffs` declare routing declaratively |
| Debug with print statements | `trace()` gives you a visual timeline |
| ~100 lines of glue code | ~10 lines of configuration |

The SDK's core idea: **describe what you want (agents, tools, handoffs) and let the framework handle how it runs.**

---

## 3. Your First Agent

An Agent needs just three things:

```
┌─────────────────────────────────┐
│           Agent                 │
├─────────────────────────────────┤
│  name          "Market Analyst" │
│  instructions  "Analyze..."     │
│  model         "gpt-4o-mini"   │
└─────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────┐
│  Runner.run(agent, user_input)  │
│  → manages the LLM call loop   │
│  → returns result.final_output  │
└─────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────┐
│  trace("label")                 │
│  → wraps execution for          │
│    observability                 │
└─────────────────────────────────┘
```

Let's create our first agent — a **Market Analyst** that we'll reuse throughout this notebook:

In [ ]:
analyst = Agent(
    name="Market Analyst",
    instructions="You analyze market trends. Given a topic, provide 3 key market insights in bullet points. Be concise.",
    model=MODEL
)

with trace("First agent run"):
    result = await Runner.run(analyst, "AI-powered compliance tools in India")
    print(result.final_output)

> **Tip:** `trace()` can wrap any `Runner.run()` call to give you a visual timeline of every LLM call, tool use, and handoff. View your traces at https://platform.openai.com/traces. We'll skip it in the rest of this notebook to reduce clutter, but it's useful for debugging.

---

## 4. @function_tool — Giving Agents Abilities

An agent without tools is just a chatbot. **Tools** let agents take actions — query databases, call APIs, do math, anything a Python function can do.

The `@function_tool` decorator turns any Python function into an agent tool. It reads the function's **name**, **docstring**, and **type hints** to auto-generate a JSON schema — the same schema OpenAI's API expects.

```
┌──────────────────────┐        ┌──────────────────────┐
│   Your Function      │  ───>  │   Tool Schema        │
│                      │        │                      │
│  @function_tool      │        │  name: "get_weather"  │
│  def get_weather(    │        │  params:              │
│    city: str         │        │    city: string       │
│  ) -> str:           │        │  description:         │
│    """Get weather.""" │        │    "Get weather."     │
└──────────────────────┘        └──────────────────────┘
```

The LLM sees the schema, decides when to call the tool, and the SDK handles execution automatically.

In [ ]:
@function_tool
def get_weather(city: str) -> str:
    """Get current weather for a city. Returns a weather summary string."""
    # In production, this would call a real weather API
    weather_data = {
        "bhubaneswar": "32°C, Partly Cloudy, Humidity: 75%",
        "delhi": "28°C, Hazy, AQI: 180",
        "bangalore": "24°C, Light Rain, Humidity: 85%",
    }
    return weather_data.get(city.lower(), f"No data available for {city}")

# The decorator auto-generates the schema
print(f"Tool name: {get_weather.name}")
print(f"Tool description: {get_weather.description}")

In [ ]:
# Give our analyst the weather tool and run it
analyst_with_tools = Agent(
    name="Market Analyst",
    instructions="You analyze market trends. When asked about a specific city, use the get_weather tool to check local conditions and incorporate that into your analysis. Provide 3 key insights in bullet points.",
    model=MODEL,
    tools=[get_weather]
)

result = await Runner.run(analyst_with_tools, "What's the business outlook for outdoor events in Bhubaneswar?")
print(result.final_output)

---

## 5. agent.as_tool() — The Manager-Employee Pattern

What if a tool's logic is too complex for a simple function? You can turn an **entire agent** into a tool for another agent.

Think of it like a **manager delegating to employees**:

```
                ┌─────────────┐
                │   Manager   │
                │   Agent     │
                └──────┬──────┘
                       │ calls as tools
              ┌────────┴────────┐
              ▼                 ▼
        ┌──────────┐     ┌──────────┐
        │ Analyst  │     │ Writer   │
        │ Agent    │     │ Agent    │
        └──────────┘     └──────────┘
```

When the manager calls `agent.as_tool()`, the employee agent runs **independently** (its own conversation, its own instructions), then returns its output as a string back to the manager. The manager stays in control throughout.

### as_tool() vs Handoffs

| | `agent.as_tool()` | `handoffs` |
|---|---|---|
| **Control** | Returns to the calling agent | Passes to the new agent permanently |
| **Analogy** | Manager asks employee, gets answer back | Relay race — pass the baton |
| **Use when** | You need the result to make further decisions | The next agent should take over completely |

In [ ]:
# Create the Email Writer — our second core agent
writer = Agent(
    name="Email Writer",
    instructions="You write professional cold emails. Given market insights, write a short (3-4 sentence) sales email that uses those insights. Sign off as 'The TechAudit Team'.",
    model=MODEL
)

# Convert our existing agents to tools
analyst_tool = analyst.as_tool(
    tool_name="market_analyst",
    tool_description="Analyze market trends for a given topic and return key insights"
)

writer_tool = writer.as_tool(
    tool_name="email_writer",
    tool_description="Write a sales email based on provided market insights"
)

# The manager orchestrates both
manager = Agent(
    name="Sales Manager",
    instructions="""You are a Sales Manager at TechAudit India (AI-powered compliance tools).

Steps:
1. Use the market_analyst tool to get insights about the prospect's industry
2. Use the email_writer tool to draft an email using those insights
3. Present the final email to the user""",
    tools=[analyst_tool, writer_tool],
    model=MODEL
)

result = await Runner.run(manager, "Prepare a sales email for a fintech company")
print(result.final_output)

### Running Agents in Parallel with asyncio.gather()

When tasks don't depend on each other, run them simultaneously. Here we reuse the **same `analyst` agent** to analyze three different markets in parallel — no need to create separate agents:

In [ ]:
# Same analyst agent, three different topics — run in parallel
topics = [
    "AI compliance tools in India",
    "AI compliance tools in Southeast Asia",
    "AI compliance tools in the Middle East",
]

results = await asyncio.gather(
    Runner.run(analyst, topics[0]),
    Runner.run(analyst, topics[1]),
    Runner.run(analyst, topics[2]),
)

for topic, r in zip(topics, results):
    print(f"\n{'='*50}")
    print(f"Topic: {topic}\n")
    print(r.final_output)

---

## 6. Handoffs — Passing the Baton

With `agent.as_tool()`, control always returns to the manager. But sometimes you want to **pass control entirely** to another agent — like a relay race where you hand off the baton.

**Handoffs** do exactly this: when Agent A hands off to Agent B, Agent B takes over the conversation completely. Agent A is done.

```
  agent.as_tool()                    handoffs
  ─────────────                      ────────
  Manager ──> Employee ──> Manager   Agent A ──> Agent B ──> (done)
  (round trip)                       (one way)
```

Use handoffs when the next agent is better suited to handle the rest of the conversation.

Let's reuse our existing `manager` and `writer` agents. We'll give the writer a `handoff_description` so the manager knows when to hand off to it:

In [ ]:
# Add a handoff_description so the manager knows when to hand off
writer.handoff_description = "Hand off to the Email Writer when you have enough context to draft a sales email"

# Create a new manager that uses handoffs instead of as_tool()
handoff_manager = Agent(
    name="Sales Manager",
    instructions="""You are a Sales Manager at TechAudit India (AI-powered compliance tools).

Steps:
1. Use the market_analyst tool to research the prospect's industry
2. Once you have enough insights, hand off to the Email Writer with the context

Do NOT write the email yourself — hand off to the specialist.""",
    tools=[analyst_tool],
    handoffs=[writer],
    model=MODEL
)

result = await Runner.run(handoff_manager, "Prepare a sales email for a healthcare company looking for audit tools")
print(f"Handled by: {result.last_agent.name}\n")
print(result.final_output)

> **Real-world pattern: Triage agents.** In production, handoffs are commonly used for customer support routing — a triage agent classifies the user's question and hands off to a specialist (billing, technical, onboarding). Each specialist has its own tools and instructions, so the triage agent doesn't need to know how to handle every case.

---

## 7. Putting It All Together — Multi-Step Workflow

Let's combine everything — `@function_tool`, `agent.as_tool()`, and `handoffs` — in one workflow using the same agents we've been building:

```
User Request
     │
     ▼
┌──────────┐  tools    ┌────────────┐
│ Research  │ ────────> │ get_weather │
│ Manager   │           └────────────┘
│           │  as_tool  ┌────────────┐
│           │ ────────> │ Analyst    │
│           │           └────────────┘
│           │
│           │  handoff  ┌────────────┐
│           │ ────────> │ Writer     │
└──────────┘           └────────────┘
```

In [ ]:
# Update the writer's instructions for briefing-style output
writer.instructions = """You write concise research briefings. You'll receive context about a city/region.
Compile all available information into a short briefing (5-6 sentences) with sections:
- Market Overview
- Local Conditions
End with a recommendation on whether to pursue the market."""

# The research manager: combines function_tool + as_tool + handoff
research_manager = Agent(
    name="Research Manager",
    instructions="""You are a research manager preparing a market briefing.

Steps:
1. Use get_weather to check local conditions
2. Use market_analyst to get industry insights
3. Hand off all gathered information to the Email Writer for a final briefing""",
    tools=[get_weather, analyst_tool],
    handoffs=[writer],
    model=MODEL
)

result = await Runner.run(research_manager, "Prepare a briefing on the compliance tech market in Bhubaneswar")
print(f"Final agent: {result.last_agent.name}\n")
print(result.final_output)

---

## 8. Key Takeaways

### Concept Map

```
Agent  ──────────────────────────────────────────────┐
  │                                                   │
  ├── Runner.run(agent, input)  → executes the agent  │
  │                                                   │
  ├── tools=[...]               → agent can call      │
  │     ├── @function_tool      → Python function      │
  │     └── agent.as_tool()     → another agent        │
  │                                                   │
  └── handoffs=[...]            → pass control         │
        └── Agent with          → takes over entirely  │
            handoff_description                        │
```

### Quick Reference

| Concept | Code | When to Use |
|---|---|---|
| **Create agent** | `Agent(name, instructions, model)` | Always — the basic building block |
| **Run agent** | `await Runner.run(agent, input)` | To execute any agent |
| **Function tool** | `@function_tool` on a function | Agent needs to call Python code |
| **Agent as tool** | `agent.as_tool(name, desc)` | Delegate subtask, get result back |
| **Parallel run** | `asyncio.gather(run1, run2, ...)` | Independent tasks, save time |
| **Handoff** | `handoffs=[agent]` | Pass control to a specialist |

> **Note:** Use `with trace("label"):` around any `Runner.run()` call to get a visual execution timeline for debugging. See Section 3 for an example.

---

## 9. Exercises

### Exercise 1: Add a Tool (Beginner)
Create a `@function_tool` called `company_lookup` that takes a company name and returns mock company data (industry, size, location). Add it to an agent and test it.

### Exercise 2: Three-Agent Pipeline (Intermediate)
Build a pipeline with three agents:
1. A **Researcher** agent (with a mock `search_database` tool) that finds relevant case studies
2. A **Strategist** agent (as_tool) that takes case studies and creates a sales strategy
3. A **Manager** agent that orchestrates the above two and then hands off to an **Email Writer** agent

Test it with: "Prepare an outreach for a healthcare company looking for compliance tools."

### Exercise 3: Customer Support Router (Intermediate)
Create a triage agent with handoffs to at least 3 specialist agents (billing, technical, onboarding). Each specialist should have relevant tools (e.g., `check_subscription_status`, `get_setup_guide`). Test with different customer queries and verify the correct specialist handles each one.

---

**What's Next?** In the next notebook, we'll explore using multiple LLM providers with LiteLLM, input guardrails, and build a deep research pipeline.

---

**Course Information:**
- **Institution:** CV Raman Global University, Bhubaneswar
- **Program:** AI Center of Excellence
- **Course:** AI Systems Engineering 2
- **Developed by:** [Poorit Technologies](https://poorit.in) - *Transform Graduates into Industry-Ready Professionals*

---